In [24]:
from dotenv import load_dotenv

# Load your API key from an environment variable (recommended)
import os
load_dotenv()
import requests
# !pip install yfinance
import pandas as pd
from io import StringIO
from pathlib import Path
from collections import defaultdict

In [25]:
import pandas as pd
import datetime

In [ ]:
API_URL = "https://stocknewsapi.com/api/v1"
# API_KEY = os.getenv("API_KEY")
API_KEY = "ABC"

In [27]:
r = requests.get("https://en.wikipedia.org/wiki/List_of_S%26P_500_companies", headers={'User-agent': 'Chrome/103.0.5060.114'}).text
df = pd.read_html(StringIO(str(r))) #load with user agent to avoid 401 error
# print(df)
TICKR_symbols = pd.DataFrame(df[0])
print(TICKR_symbols['Symbol'])

0       MMM
1       AOS
2       ABT
3      ABBV
4       ACN
       ... 
498     XYL
499     YUM
500    ZBRA
501     ZBH
502     ZTS
Name: Symbol, Length: 503, dtype: object


In [28]:
# was debugging. Please ignore.
TICKR_symbols.index[TICKR_symbols['Symbol'] == "WY"].tolist()

[491]

In [29]:
TICKR_symbols = TICKR_symbols[492:503]
print(TICKR_symbols)

    Symbol               Security             GICS Sector  \
492    WSM  Williams-Sonoma, Inc.  Consumer Discretionary   
493    WMB     Williams Companies                  Energy   
494    WTW   Willis Towers Watson              Financials   
495   WDAY          Workday, Inc.  Information Technology   
496   WYNN           Wynn Resorts  Consumer Discretionary   
497    XEL            Xcel Energy               Utilities   
498    XYL             Xylem Inc.             Industrials   
499    YUM            Yum! Brands  Consumer Discretionary   
500   ZBRA     Zebra Technologies  Information Technology   
501    ZBH          Zimmer Biomet             Health Care   
502    ZTS                 Zoetis             Health Care   

                                GICS Sub-Industry      Headquarters Location  \
492                         Homefurnishing Retail  San Francisco, California   
493            Oil & Gas Storage & Transportation            Tulsa, Oklahoma   
494                        

In [30]:
start_date = datetime.date(2022, 1, 1)
start_date_str = start_date.strftime('%m%d%Y')

end_date = datetime.date.today()
end_date_str = end_date.strftime('%m%d%Y')

In [31]:
def fetch_news_article(TICKER):
    all_articles = []
    current_page = 1

    while True:
        # Use a dictionary for request parameters - much cleaner!
        params = {
            'tickers': TICKER,
            'date': f"{start_date_str}-{end_date_str}",
            'items': 100,         # Request the maximum number of items
            'page': current_page, # Specify the page number
            'token': API_KEY,
            # 'sort': 'rank',
            # 'channels': 'earnings',  # Filter for earnings-related news
            'type': 'article' # Ensure we only get articles
        }

        # Make the request using the 'params' argument
        response = requests.get(API_URL, params=params)

        if response.status_code != 200:
            print(f"Error fetching page {current_page}. Status Code: {response.status_code}")
            print(f"Response: {response.text}")
            break

        data = response.json()

        # If the 'data' key exists and contains articles, add them to our list
        articles_on_page = data.get('data', [])
        if not articles_on_page:
            print(f"No articles found on page {current_page}. Stopping further requests.")
            break

        all_articles.extend(articles_on_page)
        print(f"Fetched page {current_page}, found {len(articles_on_page)} articles.")
        current_page += 1

        if current_page > 100:  # Limit to 100 pages to avoid excessive requests
            # will manually check that the limit of 100 pages is not crossed for any TICKR symbol
            print("Reached the limit of 100 pages. Stopping further requests.")
            break

    print("\n--- Total Articles Fetched ---")
    print(f"Total articles retrieved: {len(all_articles)}")

    # You can now work with the 'all_articles' list
    # For example, print the title of the first article if it exists
    if all_articles:
        print(f"\nExample article title: {all_articles[0]['title']}")
    return all_articles

In [32]:
out_file = Path("/Users/shreya/Documents/UCDavis/BAX_423_Big_Data_Analytics/Final_Project/Summer_Quarter/Updated_EmailProf_27Aug/news_articles_2022_25VTestV2.xlsx")
# first_write = True
for index, each_row in TICKR_symbols.iterrows():
    sheet_name = str(each_row['Symbol'])
    try:
        all_articles = fetch_news_article(sheet_name)
        all_articles = pd.DataFrame(all_articles)
        # mode = "w" if first_write else "a"
        # if mode == "w":
        #     with pd.ExcelWriter(out_file, engine="openpyxl", mode=mode) as writer:
        #         all_articles.to_excel(writer, sheet_name=sheet_name, index=False)
        # else:
        with pd.ExcelWriter(out_file, engine="openpyxl", mode="a", if_sheet_exists="new") as writer:
            all_articles.to_excel(writer, sheet_name=sheet_name, index=False)
        # first_write = False
    except Exception as e:
        print(f"Error for symbol {each_row['Symbol']}: {e}")

Fetched page 1, found 100 articles.
Fetched page 2, found 100 articles.
Fetched page 3, found 100 articles.
Fetched page 4, found 100 articles.
Fetched page 5, found 100 articles.
Fetched page 6, found 100 articles.
Fetched page 7, found 100 articles.
Fetched page 8, found 22 articles.
No articles found on page 9. Stopping further requests.

--- Total Articles Fetched ---
Total articles retrieved: 722

Example article title: Rising Tariffs Challenge Williams-Sonoma But Diversified Model Supports Growth, Says Analyst
Fetched page 1, found 100 articles.
Fetched page 2, found 100 articles.
Fetched page 3, found 100 articles.
Fetched page 4, found 100 articles.
Fetched page 5, found 100 articles.
Fetched page 6, found 65 articles.
No articles found on page 7. Stopping further requests.

--- Total Articles Fetched ---
Total articles retrieved: 565

Example article title: Williams CEO to Present at 2025 Barclays CEO Energy-Power Conference
Fetched page 1, found 100 articles.
Fetched page 2, 